# Intro til Machine Learning — 1: Neurale netværk

I regressionsforløbet trænede I jeres første lille netværk: ét neuron (`nn.Linear`) med autograd og Adam, og I mødte træningsloopets rytme — nulstil gradienter, `.backward()`, tag et skridt. Et enkelt neuron finder kun den bedste **linje**, men verden er sjældent en linje:
nogle mønstre er krøllede, buede og indviklede. Løsningen er at stable mange små
linje-byggeklodser oven på hinanden med lidt "krølning" imellem — og så har man et
**neuralt netværk**, den model der har drevet AI-revolutionen.

I denne notebook:
1. **Neuronen** — netværkets mindste byggesten (afsnit 1)
2. **`nn.Module`** — netværk skrives som *klasser* (jeres nye klasse-viden i aktion!) (afsnit 2)
3. **Træningsloopet** — vi træner et netværk til at spotte legendariske Pokémon (afsnit 3)

> **Om opgaverne:** Der er med vilje flere opgaver, end I kan nå — ingen forventes at nå alt. Opgaver mærket **(ekstra)** er til jer, der er foran; opgaver mærket **(find fejlen)** har en bevidst fejl, som I skal finde og rette (så en fejl dér er meningen).

## Setup

In [ ]:
# Henter data + plottehjælper fra GitHub (Plan B: upload dem manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/Pokemon.csv
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from helpers import plot_decision_boundary

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("Pokemon.csv")
df.head(3)

# 1: Fra linje til neuron

En **neuron** (opkaldt efter hjernecellerne, meget løst) er bare en lille regnemaskine, der:

1. tager nogle input-tal $x_1, x_2, \dots$
2. ganger hvert input med sin egen **vægt** $w_1, w_2, \dots$ og lægger en **bias** $b$ til:
$$z = w_1 x_1 + w_2 x_2 + \dots + b$$
3. (og som regel: sender resultatet gennem en **aktiveringsfunktion** — dem dykker vi ned i i notebook 3)

Genkender I trin 2? **Det er jo bare linjens ligning med flere variable!** En neuron med ét
input er bogstaveligt talt $z = ax + b$. Så her er dagens plot-twist: `nn.Linear(1, 1)` **er** en neuron. Regressionen, I lavede, var et neuralt netværk med én neuron.
Surprise — I har allerede trænet jeres første netværk.

`nn.Linear` starter med tilfældige vægte (derfor `torch.manual_seed`!), men vi kan også
sætte dem selv:

In [ ]:
neuron = nn.Linear(1, 1)     # 1 input → 1 output
print("tilfældig vægt:", neuron.weight.item())
print("tilfældig bias:", neuron.bias.item())

with torch.no_grad():        # vi piller ved vægtene manuelt — det skal autograd ikke se
    neuron.weight[0, 0] = 2.0
    neuron.bias[0] = 1.0

x = torch.tensor([[3.0]])    # NB: nn.Linear vil have 2D-input: (antal eksempler, antal features)
print("neuron(3) =", neuron(x).item(), "  (tjek: 2 · 3 + 1 = 7)")

## Mange neuroner = et lag

`nn.Linear(6, 4)` er **4 neuroner**, der hver især kigger på de samme 6 input og laver
deres egen vægtede sum. Vægtene samles i en matrix med shape `(4, 6)` — én række pr.
neuron, én kolonne pr. input:

In [ ]:
layer = nn.Linear(6, 4)
print("vægte:", layer.weight.shape)   # (4 neuroner, 6 input)
print("bias:  ", layer.bias.shape)    # én bias pr. neuron

otte_pokemon = torch.randn(8, 6)    # 8 tilfældige "Pokémon" med 6 stats
print("output:", layer(otte_pokemon).shape)   # 8 eksempler ind → 8 gange 4 tal ud

## Krølningen: en hurtig smagsprøve på aktiveringsfunktioner

Stabler man to `nn.Linear`-lag direkte oven på hinanden, får man... stadig bare en lineær
funktion (mere om hvorfor i opgave 1.6). Tricket, der giver netværk deres kraft, er at
putte en **ikke-lineær funktion** ind mellem lagene.

Den mest berømte er **sigmoid**: $\sigma(z) = \frac{1}{1+e^{-z}}$. Den maser ethvert tal
ind i intervallet $(0, 1)$ — store tal → tæt på 1, små (negative) tal → tæt på 0. Perfekt,
når svaret skal være en **sandsynlighed** (fx "hvor sikker er jeg på, at den her er
legendarisk?"):

In [ ]:
z = torch.tensor([-5.0, -1.0, 0.0, 1.0, 5.0])
print(torch.sigmoid(z))   # alt bliver klemt ind mellem 0 og 1

### Opgaver

##### Opgave 1.1
En neuron har vægtene $w_1 = 0{,}5$, $w_2 = -1$ og bias $b = 2$, og får input
$x_1 = 4$, $x_2 = 3$. Regn først neuronens output $z = w_1 x_1 + w_2 x_2 + b$ **i hånden**
— og udfyld så formlen i koden og tjek dit facit.

In [ ]:
w1, w2, b = 0.5, -1.0, 2.0
x1, x2 = 4.0, 3.0
z = ...
print(z)

##### Opgave 1.2
Cellen laver `nn.Linear(3, 1)`. Ændr den til `nn.Linear(6, 4)` og derefter `nn.Linear(4, 1)`
— men **forudsig `weight.shape` FØR I kører** hver gang. Reglen: shape er
`(antal output, antal input)`.

In [ ]:
layer = nn.Linear(3, 1)
print(layer.weight.shape)

##### Opgave 1.3
Regressionsforløbet fandt en tilsvarende bedste linje $y \approx 0{,}74 \cdot x + 0$ for
Attack → Total. Sæt neuronens vægt og bias til de tal og tegn dens forudsigelser oven på
punktskyen — tegner én neuron præcis den samme linje som regressionen?

In [ ]:
x_attack = torch.tensor(df["Attack"].values, dtype=torch.float32)
y_total = torch.tensor(df["Total"].values, dtype=torch.float32)
x_attack = (x_attack - x_attack.mean()) / x_attack.std()
y_total = (y_total - y_total.mean()) / y_total.std()

neuron = nn.Linear(1, 1)
with torch.no_grad():
    neuron.weight[0, 0] = ...
    neuron.bias[0] = ...

with torch.no_grad():
    y_hat = neuron(x_attack.reshape(-1, 1)).squeeze()   # reshape: 800 tal → (800, 1)

plt.scatter(x_attack, y_total, alpha=0.3)
plt.plot(x_attack, y_hat, color="crimson", linewidth=2)
plt.xlabel("Attack (std)")
plt.ylabel("Total (std)")
plt.show()

##### Opgave 1.4 (find fejlen)
Cellen fodrer et lag med Pokémon-data, men PyTorch protesterer højlydt. Læs fejlbeskeden —
den fortæller, hvilke shapes der støder sammen — og ret **laget** (ikke dataene), så det
passer til input med 5 features.

In [ ]:
fem_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def"]   # 5 features
X5 = torch.tensor(df[fem_stats].values, dtype=torch.float32)
print("input shape:", X5.shape)

layer = nn.Linear(6, 1)
print(layer(X5))

##### Opgave 1.5
Neuronen nedenfor har sigmoid på outputtet. Kør cellen, og fjern så `torch.sigmoid(...)`
(behold bare `z`) og kør igen. Sammenlign de to output-intervaller — hvornår ville man
ønske sig hvilket?

In [ ]:
neuron = nn.Linear(6, 1)
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = torch.tensor(df[seks_stats].values, dtype=torch.float32)
X = (X - X.mean(dim=0)) / X.std(dim=0)

with torch.no_grad():
    z = neuron(X).squeeze()
    ud = torch.sigmoid(z)     # ← prøv også bare: ud = z
print("mindste output:", ud.min().item())
print("største output:", ud.max().item())

##### Opgave 1.6 (ekstra)
Tag to lineære funktioner, fx $f(x) = 2x + 1$ og $g(x) = 3x - 2$, og sæt dem sammen:
regn $g(f(x))$ ud på papir. Hvilken slags funktion får I? Hvad fortæller det om at stable
`nn.Linear`-lag **uden** noget imellem?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.7
Udfyld tallene, så laget tager **6 stats ind** og giver **3 neuroner** ud. Tjek formen bagefter.

In [ ]:
layer = nn.Linear(..., ...)      # ← 6 ind, 3 ud
print(layer.weight.shape)        # forventet: (3, 6) — 3 neuroner, hver med 6 vægte

##### Opgave 1.8 (find fejlen)
Cellen skal sende tre tal gennem en neuron, men den crasher med en fejl om, at former
ikke passer sammen. Find og ret det ene tal, der er galt.

In [ ]:
neuron = nn.Linear(1, 1)                 # forventer 1 tal ind
x = torch.tensor([1.0, 2.0, 3.0])        # ...men her kommer tre!
print(neuron(x))

##### Opgave 1.9 (ekstra)
Byg et bittelille netværk i hånden: send en Pokémons seks stats gennem et lag og krøl
resultatet med sigmoid. Udfyld kaldet, der sender `x` gennem laget.

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
x = torch.tensor(df[seks_stats].iloc[0].values, dtype=torch.float32)
layer = nn.Linear(6, 4)
hidden = torch.sigmoid(...)      # ← send x gennem laget
print("fire skjulte neuroner:", hidden)

# 2: `nn.Module` — netværk er klasser

Nu skal lagene stables til et rigtigt netværk — og her kommer jeres **klasse**-viden fra
intro-programmering i spil, for i PyTorch skriver man netværk som klasser. Kort recap:

- `class MinKlasse:` definerer en skabelon med **attributter** (data) og **metoder** (funktioner).
- `__init__(self)` køres, når man opretter et objekt — her gemmer man attributterne på `self`.
- `class MinKlasse(AndenKlasse):` betyder **nedarvning**: klassen får alt, hvad `AndenKlasse` kan, gratis.

(Rusten? Kig tilbage i afsnit P13 i [Intro-Programmering, notebook 2](../Intro-Programmering/3-Plots-klasser-og-tekst.ipynb).)

Et PyTorch-netværk er en klasse, der nedarver fra `nn.Module` og udfylder to ting:

1. **`__init__`**: hvilke lag har netværket? (byggeklodserne)
2. **`forward`**: i hvilken rækkefølge skal data flyde igennem dem? (samlevejledningen)

Mellem lagene bruger vi aktiveringsfunktionen **ReLU** — den er endnu simplere end sigmoid:
negative tal bliver til 0, positive tal passerer urørt. Mere om *hvorfor* i notebook 3;
indtil videre er ReLU bare vores "krølning" mellem lagene.

In [ ]:
class MyNetwork(nn.Module):
    def __init__(self):
        super().__init__()                  # tænd for alt nn.Module-maskineriet — ALDRIG glemme!
        self.layer1 = nn.Linear(6, 16)        # 6 stats ind → 16 skjulte neuroner
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)        # 16 skjulte → 1 output

    def forward(self, x):                   # sådan flyder data gennem netværket
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = MyNetwork()
print(model)

De 16 neuroner i midten kaldes et **skjult lag** (skjult, fordi hverken input eller
output ser dem direkte). Antallet — 16 — er et frit valg: flere neuroner = klogere netværk,
men også flere parametre at træne.

Hvor mange parametre (vægte + bias'er) har vores netværk egentlig? `p.numel()` tæller
tallene i én parameter-tensor, og `model.parameters()` går alle igennem:

In [ ]:
antal = sum(p.numel() for p in model.parameters())
print("antal parametre:", antal)
# tjek i hånden: lag1: 6·16 + 16 = 112, lag2: 16·1 + 1 = 17 → 129 i alt

Og netværket kan bruges med det samme — kald det som en funktion (det kalder `forward`
bag kulisserne). Uden træning er outputtet naturligvis rent nonsens fra tilfældige vægte:

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = torch.tensor(df[seks_stats].values, dtype=torch.float32)
X = (X - X.mean(dim=0)) / X.std(dim=0)      # standardisering — I kender rytmen

with torch.no_grad():
    print(model(X[:5]).squeeze())            # 5 utrænede "gæt" — nonsens, men FORMEN er rigtig

### Opgaver

##### Opgave 2.1
Netværket ovenfor har 16 skjulte neuroner. Lav to nye udgaver: én med **4** og én med
**64** skjulte neuroner (ret begge `nn.Linear`-linjer, så dimensionerne stadig passer!),
og print antal parametre for hver. Hvor meget voksede netværket?

In [ ]:
class MyNetwork4(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)    # ← ret 16 til 4 (og lav bagefter en 64-udgave)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

print(sum(p.numel() for p in MyNetwork4().parameters()))

##### Opgave 2.2
Udfyld `forward`-metoden: data skal flyde **lag1 → aktivering → lag2**.

In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)

    def forward(self, x):
        x = ...
        x = ...
        x = ...
        return x

model = Network()
with torch.no_grad():
    print(model(X[:3]).squeeze())   # virker den? så skal der komme 3 tal ud

##### Opgave 2.3 (find fejlen)
Klassen nedenfor crasher allerede, når man prøver at **oprette** modellen. Kør cellen, læs
fejlbeskeden, og brug jeres klasse-viden: hvad er det for en vigtig linje, der mangler i
`__init__`?

In [ ]:
class BrokenNetwork(nn.Module):
    def __init__(self):
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model = BrokenNetwork()

##### Opgave 2.4 (find fejlen)
Denne klasse kan godt oprettes — men når man KALDER modellen, brokker PyTorch sig over, at
`forward` ikke er implementeret. Men den står der da...?! Kig meget nøje på metodens navn.

In [ ]:
class MysteryNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)

    def Forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model = MysteryNetwork()
print(model(X[:3]))

##### Opgave 2.5
Tilføj et **ekstra skjult lag** til netværket, så data flyder 6 → 16 → 8 → 1. Husk reglen
fra opgave 1.4: hvert lags input-antal skal matche det forrige lags output-antal — som
togskinner, der skal passe sammen.

In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        # tilføj lag her — og opdater forward!

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = Network()
print(model)

##### Opgave 2.6 (ekstra)
Byg et netværk med arkitekturen **6 → 32 → 16 → 1** ved at udfylde alle dimensionerne.

In [ ]:
class BigNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(..., ...)
        self.layer2 = nn.Linear(..., ...)
        self.layer3 = nn.Linear(..., ...)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        x = self.layer3(x)
        return x

model = BigNetwork()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

##### Opgave 2.7
Kør cellen nedenfor 3-4 gange. Den opretter et NYT netværk hver gang og lader det gætte på
de samme 3 Pokémon — men gættene er nye hver gang! Hvorfor? Og hvorfor gav vores celler
tidligere i notebooken samme resultat hver gang, selvom vægtene var "tilfældige"?

In [ ]:
nyt_network = MyNetwork()
with torch.no_grad():
    print(nyt_network(X[:3]).squeeze())

##### Opgave 2.8
Kig på `MitNetvaerk`-klassen. Hvad har **vi selv** skrevet, og hvad **arver** vi gratis fra
`nn.Module`? (Tip: vi har fx aldrig skrevet koden, der gør, at `model(X)` virker, eller at
`model.parameters()` kan finde alle vægtene...)

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

# 3: Træningsloopet — er den Pokémon legendarisk?

Tid til at samle det hele! Missionen: træn et netværk, der ud fra de seks kamp-stats
afgør, om en Pokémon er **legendarisk**. Det er **binær klassifikation** — svaret er
0 (almindelig) eller 1 (legendarisk) — så opskriften er:

- Output-laget skal give **én** sandsynlighed → 1 output-neuron med **sigmoid** til sidst.
- Tabsfunktionen skal måle "hvor forkert er en sandsynlighed?" → **`nn.BCELoss`**
 (*binary cross entropy* — den straffer hårdt, når modellen er *selvsikker og forkert*).

## Trin 1: Train/test-split — eksamensreglen

Vi deler dataene i **træningssæt** (80 %) og **testsæt** (20 %). Modellen må KUN lære af
træningssættet — testsættet er gemt væk til den afsluttende eksamen. At måle på data,
modellen har trænet på, svarer til at give eleverne facitlisten med til eksamen: flotte
karakterer, nul viden.

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)          # standardisering, som altid
y_np = df["Legendary"].astype(int).values.astype("float32")   # True/False → 1.0/0.0

X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

print("træning:", X_train.shape, "| test:", X_test.shape)
print("andel legendariske i træningssættet:", round(y_train.mean().item(), 3))

## Trin 2: Modellen

6 stats ind → 16 skjulte neuroner (ReLU) → 1 sandsynlighed ud (sigmoid):

In [ ]:
class LegendarySpotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        x = self.sigmoid(x)      # output bliver en sandsynlighed mellem 0 og 1
        return x

model = LegendarySpotter()
print(model)

## Trin 3: Træningsloopet

Præcis samme rytme som i regressionsforløbet — læg mærke til de fem nummererede trin, for de er
**altid** de samme. Ny ven: optimizeren **Adam**, en smartere udgave af SGD, der løbende
justerer skridtlængden selv. Den er standardvalget i praksis:

In [ ]:
model = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []

for epoch in range(500):
    # 1. Nulstil gradienter
    optimizer.zero_grad()
    # 2. Forward: modellens gæt for alle trænings-Pokémon
    y_hat = model(X_train).squeeze()
    # 3. Regn tabet: hvor forkerte er gættene?
    loss = loss_fn(y_hat, y_train)
    # 4. Backward: find alle gradienter
    loss.backward()
    # 5. Tag ét skridt ned ad bakken
    optimizer.step()

    loss_history.append(loss.item())   # .item(): træk TALLET ud af tensoren

print("sluttab:", round(loss_history[-1], 4))

plt.plot(loss_history)
plt.xlabel("epoke")
plt.ylabel("BCE-tab")
plt.title("Tabskurven — den skal falde og flade ud")
plt.show()

## Trin 4: Eksamen — hvor god er den på testsættet?

Modellen svarer med sandsynligheder. Vi runder af ved 0,5 ("tror du mest ja eller mest
nej?") og sammenligner med de rigtige svar. Andelen af rigtige gæt kaldes **accuracy** —
og husk tricket fra opgave 3.8: gennemsnittet af en True/False-række er netop andelen af True:

In [ ]:
with torch.no_grad():
    probabilities = model(X_test).squeeze()

pred = (probabilities > 0.5).float()          # sandsynlighed → 0 eller 1
accuracy = (pred == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

Pæn høj accuracy!...eller hvad?

## Fælden: den dovne model

Kun ca. 8 % af alle Pokémon er legendariske. En "model", der ALTID svarer "ikke
legendarisk" — uden at kigge på så meget som én stat — ville altså ramme rigtigt ca. 92 %
af gangene. **Høj accuracy er billig, når klasserne er skæve!** Det tjekker vi i opgave
10.6, og i opgave 3.5-10.8 finder vi ud af, hvad modellen egentlig har lært.

### Opgaver

##### Opgave 3.1
Skabelonen nedenfor er træningsloopet med fem huller — ét pr. nummereret trin. Udfyld dem
(kig IKKE i cellen ovenfor... okay, kig lidt hvis I sidder fast ).

In [ ]:
model_a = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_a.parameters(), lr=0.01)

for epoch in range(500):
    # 1. Nulstil gradienter
    ...
    # 2. Forward: modellens gæt
    y_hat = ...
    # 3. Regn tabet
    loss = ...
    # 4. Backward: find gradienterne
    ...
    # 5. Tag ét skridt
    ...

print("sluttab:", round(loss.item(), 4))

##### Opgave 3.2
Skru på `lr` og antal epoker i træningsloopet, og mål test-accuracy hver gang (genbrug
eksamens-cellen). Hvor højt kan I komme — kan I slå 95 %? Notér jeres bedste resultat, og
kør jeres bedste opskrift to gange: får I samme tal?

In [ ]:
model_b = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_b.parameters(), lr=0.01)   # ← skru her
for epoch in range(500):                                       # ← og her
    optimizer.zero_grad()
    y_hat = model_b(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = (model_b(X_test).squeeze() > 0.5).float()
print(f"test-accuracy: {(pred == y_test).float().mean().item():.1%}")

##### Opgave 3.3 (find fejlen)
Loopet nedenfor kører uden fejlbeskeder... men tabet rokker sig ikke ud af stedet, og
accuracy er elendig. Rytmen er kommet i uorden: kig på rækkefølgen af de fem trin og ret
den. (Hvad SKAL der være sket, før `step()` kan tage et fornuftigt skridt? Og hvornår må
man tidligst nulstille?)

In [ ]:
model = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(500):
    y_hat = model(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    optimizer.step()          # skridt...?
    loss.backward()            # ...før gradienterne er regnet?
    optimizer.zero_grad()     # ...og så slettes de nye gradienter?!

print("sluttab:", round(loss.item(), 4), "— det rokker sig ikke!")

##### Opgave 3.4 (find fejlen)
Nogen ville gemme tabskurven, men plottet crasher med en fejl om "requires grad". Kig på
linjen med `append` — hvad mangler der? (Tip: hvad gør `.item()`, og hvorfor brugte vi den
i det rigtige loop?)

In [ ]:
model = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []

for epoch in range(200):
    optimizer.zero_grad()
    y_hat = model(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(loss)      # hmm...

plt.plot(loss_history)
plt.show()

##### Opgave 3.5
Træn tre modeller med læringsraterne **0.001, 0.05 og 1.0** og plot deres tre tabskurver
i samme figur (skabelonen er klar — den mangler bare `label` og `plt.legend()`). Sæt ord
på hver kurve: tålmodig? effektiv? kaotisk?

In [ ]:
for lr in [0.001, 0.05, 1.0]:
    model_e = LegendarySpotter()
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model_e.parameters(), lr=lr)
    history = []
    for epoch in range(300):
        optimizer.zero_grad()
        loss = loss_fn(model_e(X_train).squeeze(), y_train)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    plt.plot(history)   # ← tilføj label=f"lr = {lr}"

# ← tilføj plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 3.6
Den dovne model: beregn accuracy for "modellen", der ALTID gætter "ikke legendarisk"
(altså gæt = 0 for alle), og sammenlign med jeres netværk. Slår jeres netværk overhovedet
den dovne? Og find så ud af det VIGTIGE tal: af de legendariske Pokémon i testsættet —
hvor mange fanger hver "model"?

In [ ]:
lazy_pred = torch.zeros(len(y_test))
lazy_accuracy = (lazy_pred == y_test).float().mean()
print(f"den dovne models accuracy: {lazy_accuracy.item():.1%}")

# jeres netværks accuracy (genbrug fra tidligere):
...

# hvor mange af testsættets legendariske fanger netværket?
# tip: kig kun på rækkerne hvor y_test == 1
...

##### Opgave 3.7
Udfyld eksamens-cellen: sandsynligheder → 0/1-gæt (grænse 0,5) → accuracy.

In [ ]:
with torch.no_grad():
    probabilities = model(X_test).squeeze()

pred = (probabilities > ...).float()
accuracy = (... == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

##### Opgave 3.8
Hvor meget kan netværket nå med kun **2** features? Træn en model, der kun ser `HP` og
`Defense` (skabelonen har allerede skåret X ned — kolonne 0 og 2). Hvad sker der med
accuracy — og vigtigere: hvor mange legendariske fanger den? Prøv bagefter med et andet
feature-par og se, hvor meget PARRET betyder.

In [ ]:
X_train2 = X_train[:, [0, 2]]    # kun HP og Defense — prøv også andre par!
X_test2 = X_test[:, [0, 2]]

class SmallSpotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(..., 16)   # ← hvor mange features ser den nu?
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.layer2(self.activation(self.layer1(x))))

model2 = SmallSpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.01)
for epoch in range(500):
    optimizer.zero_grad()
    loss = loss_fn(model2(X_train2).squeeze(), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = (model2(X_test2).squeeze() > 0.5).float()
print(f"accuracy med 2 features: {(pred == y_test).float().mean().item():.1%}")
print("fangede legendariske:", int(pred[y_test == 1].sum()), "af", int((y_test == 1).sum()))

##### Opgave 3.9 (ekstra)
To identiske netværk, to identiske loops — men det ene træner på de RÅ stats (uden
standardisering) med SGD, det andet på de standardiserede. Kør cellen og sammenlign.
Payoff for regressionsforløbet! (Bemærk: vi bruger SGD her; prøv bagefter med Adam og se, at den
delvist redder situationen — men stadig ikke helt.)

In [ ]:
X_raw = torch.tensor(df[seks_stats].values.astype("float32"))
X_raw_train, X_raw_test, _, _ = train_test_split(X_raw.numpy(), y_np, test_size=0.2, random_state=42)
X_raw_train = torch.tensor(X_raw_train)

for name, data in [("standardiseret", X_train), ("rå tal", X_raw_train)]:
    torch.manual_seed(0)
    model_t = LegendarySpotter()
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.SGD(model_t.parameters(), lr=0.1)
    for epoch in range(500):
        optimizer.zero_grad()
        loss = loss_fn(model_t(data).squeeze(), y_train)
        loss.backward()
        optimizer.step()
    print(f"{name}: sluttab = {round(loss.item(), 4)}")

##### Opgave 3.10 (ekstra)
Byg klassifikationsopgaven om til **regression**: forudsig `HP` ud fra de fem ANDRE stats.
Tre ting skal ændres i forhold til LegendeSpotter-opskriften: (1) output-aktiveringen
(sigmoid?! til HP-værdier på 20-255?), (2) tabsfunktionen, (3) målingen til sidst (accuracy
giver ikke mening for kommatal — brug fx den gennemsnitlige fejl i HP-point).

In [ ]:
fem = ["Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_hp = df[fem].values.astype("float32")
X_hp = (X_hp - X_hp.mean(axis=0)) / X_hp.std(axis=0)
y_hp = df["HP"].values.astype("float32")

X_hp_train, X_hp_test, y_hp_train, y_hp_test = train_test_split(X_hp, y_hp, test_size=0.2, random_state=42)
X_hp_train, X_hp_test = torch.tensor(X_hp_train), torch.tensor(X_hp_test)
y_hp_train, y_hp_test = torch.tensor(y_hp_train), torch.tensor(y_hp_test)

# byg model + loop her — start fra LegendeSpotter-opskriften og lav de tre ændringer
...

##### Opgave 3.11
Hvorfor snyder man sig selv, hvis man måler accuracy på **træningsdataene**? Og hvad tror
I, der sker med (a) accuracy på træningssættet og (b) accuracy på testsættet, hvis man
lader et STORT netværk træne i ekstremt mange epoker på et LILLE datasæt?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*